In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras import layers, Sequential
from utils import preprocess

load and preprocess data

In [ ]:
# DATA
image_data = []

df = pd.read_csv("dataset2/driving_log.csv")

image_paths = df.iloc[:,0].values
steering_angles = df.iloc[:,3].values



In [ ]:
image_paths.shape

In [ ]:
# loading images using image path from df
for i in range(len(image_paths)):
    img = cv2.imread(image_paths[i])
    image_data.append(img)

In [ ]:
# After loading data, before train_test_split
threshold = 0.05  # Define "straight" as angles between -0.05 and 0.05
straight_indices = np.where(np.abs(steering_angles) < threshold)[0]
turn_indices = np.where(np.abs(steering_angles) >= threshold)[0]

# Keep all turning samples, but only keep 50% of straight samples
keep_straight = np.random.choice(straight_indices, size=int(len(straight_indices) // 2), replace=False)
keep_indices = np.concatenate([turn_indices, keep_straight])

# Filter data
balanced_images = [image_data[i] for i in keep_indices]
balanced_angles = steering_angles[keep_indices]

# Then do train_test_split on balanced data
X_train, X_test, y_train, y_test = train_test_split(balanced_images, balanced_angles, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(balanced_angles, bins=100, edgecolor='black')
plt.xlabel('Steering Angle')
plt.ylabel('Frequency')
plt.title('Distribution of Steering Angles')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(image_data, steering_angles, random_state=42, shuffle=True, test_size=0.2)

In [ ]:
def augment_flip(image, steering_angle):
    flipped_image = cv2.flip(image, 1) # flip horizontally
    flipped_angle = -steering_angle
    return flipped_image, flipped_angle

def augment_brightness(image):
    brightness_factor = np.random.uniform(0.6, 1.2)
    adjusted = (image * brightness_factor).clip(0, 255).astype('uint8')
    return adjusted

def augment_zoom(image):
    zoom = np.random.uniform(1,1.3)
    h, w = image.shape[:2]
    cx, cy = w // 2, h//2

    M = cv2.getRotationMatrix2D((cx, cy), 0, zoom)

    zoomed = cv2.warpAffine(image, M, (w, h))

    return zoomed

def augment_panning(image, pan_range=0.075):
    h, w = image.shape[:2]  
    tx = np.random.uniform(-pan_range, pan_range) * w
    
    M = np.float32([[1, 0, tx], [0, 1, 0]])

    panned = cv2.warpAffine(image, M, (w, h))

    return panned

In [ ]:
# Demonstrate each augmentation
sample_img = image_data[100]  # Pick a sample image

# Apply augmentations
flipped_img, _ = augment_flip(sample_img, 0)
brightness_img = augment_brightness(sample_img)
zoomed_img = augment_zoom(sample_img)
panned_img = augment_panning(sample_img)

# Create plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original image
axes[0, 0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

# Flipped
axes[0, 1].imshow(cv2.cvtColor(flipped_img, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('Horizontal Flip')
axes[0, 1].axis('off')

# Brightness adjusted
axes[0, 2].imshow(cv2.cvtColor(brightness_img, cv2.COLOR_BGR2RGB))
axes[0, 2].set_title('Brightness Adjustment')
axes[0, 2].axis('off')

# Zoomed
axes[1, 0].imshow(cv2.cvtColor(zoomed_img, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Zoom (1.0-1.3x)')
axes[1, 0].axis('off')

# Panned
axes[1, 1].imshow(cv2.cvtColor(panned_img, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title('Horizontal Panning')
axes[1, 1].axis('off')

# Hide the last subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def augment_image(image, steering_angle):
    if np.random.random() > 0.5:
        image, steering_angle = augment_flip(image, steering_angle)
    if np.random.random() > 0.5:
        image = augment_brightness(image)
    if np.random.random() > 0.5:
        image = augment_zoom(image)
    if np.random.random() > 0.5:
        image = augment_panning(image)
    
    return image, steering_angle

In [ ]:
# augment 30% of training images 
X_train = np.array(X_train)
y_train = np.array(y_train)

augment_ratio = 0.2
num_to_augment = int(X_train.shape[0]) * augment_ratio

indices_to_augment = np.random.choice(X_train.shape[0], int(num_to_augment), replace=False)

for idx in indices_to_augment:
    X_train[idx], y_train[idx] = augment_image(X_train[idx], y_train[idx])

In [ ]:
X_train_processed = np.array([preprocess(img) for img in X_train])
X_test_processed = np.array([preprocess(img) for img in X_test])

In [ ]:
# Display samples
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sample from training data
sample_train_idx = 0
axes[0].imshow(X_train_processed[sample_train_idx])
axes[0].set_title(f'Processed Training Sample\nSteering Angle: {y_train[sample_train_idx]:.4f}')
axes[0].axis('off')

# Sample from testing data
sample_test_idx = 0
axes[1].imshow(X_test_processed[sample_test_idx])
axes[1].set_title(f'Processed Test Sample\nSteering Angle: {y_test[sample_test_idx]:.4f}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")

In [ ]:
# build model
model = Sequential([
    layers.Conv2D(24, (5,5), strides=(2,2), activation='relu', input_shape=(66, 200, 3)),
    layers.Conv2D(36, (5, 5), strides=(2, 2), activation='relu'),
    layers.Conv2D(48, (5, 5), strides=(2, 2), activation='relu'),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(1164, activation='relu'),
    layers.Dense(100, activation='relu'),
    layers.Dense(50, activation='relu'),
    layers.Dense(10, activation='relu'),
    layers.Dense(1)  # Output: steering angle
])

In [ ]:
# compile
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
# sample_weights = np.where(np.abs(y_train) < 0.05, 0.5, 2.0)

In [ ]:
# train
h = model.fit(X_train_processed, y_train, 
              validation_data=(X_test_processed, y_test),
              epochs=10,
              )

In [ ]:
# EVALUATE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(h.history['loss'], label='train loss')
ax1.plot(h.history['val_loss'], label='validation loss')
ax1.set_title('Model Loss (MSE)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(h.history['mae'], label='train MAE')
ax2.plot(h.history['val_mae'], label='validation MAE')
ax2.set_title('Model Mean Absolute Error (MAE)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()

plt.tight_layout()
plt.show()

model.save("new_model.keras")